In [1]:
import matplotlib.pyplot as plt 
import numpy as np 
import os 
import pandas as pd # 

In [2]:
df = pd.read_csv('match_data_50_tourns_modified.csv')

In [3]:
df.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,242,32,17,32,17,5,0,0.0,1.000000,5808
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,4,245,118,808,430,2,5,1.0,0.285714,5808
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,215,101,49,336,147,5,3,0.0,0.625000,5808
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,282,91,35,235,97,5,2,0.0,0.714286,5808
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,61,281,170,1112,649,2,5,1.0,0.285714,5808


In [4]:
n = round(len(df) / 6)
df_holdout = df.tail(n)
df=df.iloc[:-n]

### Linear regression with player statistical features
Here we pick only the statistical features for linear regression, elo ratings and match and frame win predictions from the elo ratings are omitted.

In [5]:
features_to_remove = ['player1', 'player2', 'player1_elo','player2_elo','elo_match_win_rate','elo_frame_win_rate',
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df.drop(columns=features_to_remove) 
y = df['win_percentage']

In [6]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit


In [7]:
tscv = TimeSeriesSplit(n_splits=5)

rmse_list = []
train_sizes = []

for train_index, test_index in tscv.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Test size:  {len(test_index)}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    rmse_list.append(rmse)
    train_sizes.append(len(train_index))

    print(f"Fold RMSE: {rmse} \n")

  Train size: 705
  Test size:  702
Fold RMSE: 0.3194242067203229 

  Train size: 1407
  Test size:  702
Fold RMSE: 0.2727147195032068 

  Train size: 2109
  Test size:  702
Fold RMSE: 0.26971674607617124 

  Train size: 2811
  Test size:  702
Fold RMSE: 0.25195251651455125 

  Train size: 3513
  Test size:  702
Fold RMSE: 0.3062056452754089 



In [8]:
weighted_rmse = np.average(rmse_list, weights=train_sizes)
print(f"Weighted Avg RMSE:   {weighted_rmse:.4f}")

Weighted Avg RMSE:   0.2809


The WRMSE for the Elo frame win rate prediction was 0.2714. Since the Elo rating itself is implicitly derived from the same player statistics we used as features, this suggests that using calculating Elo rating and using it to predict win percentage provides a more effective formulation than applying linear regression directly to those statistics.

### Linear regression with statistical features based on player statistic differences
Intuitively, one can argue that differences in player statistics—such as match win ratio or frame win ratio—may be sufficient for the prediction scenario that we have. However, this approach overlooks the role of experience. For example, a player who has achieved the same win ratio as another but has played significantly more games may perform better due to their greater experience.

In [9]:
print(df.columns)

Index(['player1', 'player2', 'best_of', 'player1_elo', 'player2_elo',
       'elo_match_win_rate', 'elo_frame_win_rate', 'p1_matches_played',
       'p1_matches_won', 'p1_frames_played', 'p1_frames_won',
       'p2_matches_played', 'p2_matches_won', 'p2_frames_played',
       'p2_frames_won', 'p1_frames_played_1_year', 'p1_frames_won_1_year',
       'p1_frames_played_3_years', 'p1_frames_won_3_years',
       'p2_frames_played_1_year', 'p2_frames_won_1_year',
       'p2_frames_played_3_years', 'p2_frames_won_3_years', 'score1', 'score2',
       'match_result', 'win_percentage', 'tournament_id'],
      dtype='object')


In [13]:
dfm = df

In [14]:
dfm['p1_matches_win_ratio']=dfm['p1_matches_won']/df['p1_matches_played']
dfm['p2_matches_win_ratio']=dfm['p2_matches_won']/df['p2_matches_played']
dfm['p1_frames_win_ratio']=dfm['p1_frames_won']/df['p1_frames_played']
dfm['p2_frames_win_ratio']=dfm['p2_frames_won']/df['p2_frames_played']
dfm['p1_frames_win_ratio_1_year']=dfm['p1_frames_won_1_year']/df['p1_frames_played_1_year']
dfm['p2_frames_win_ratio_1_year']=dfm['p2_frames_won_1_year']/df['p2_frames_played_1_year']
dfm['p1_frames_win_ratio_3_years']=dfm['p1_frames_won_3_years']/df['p1_frames_played_3_years']
dfm['p2_frames_win_ratio_3_years']=dfm['p2_frames_won_3_years']/df['p2_frames_played_3_years']



In [15]:
dfm.fillna(0.5, inplace=True)

In [16]:
dfm['matches_win_ratio_diff']=dfm['p1_matches_win_ratio']-dfm['p2_matches_win_ratio']
dfm['frames_win_ratio_diff']=dfm['p1_frames_win_ratio']-dfm['p2_frames_win_ratio']
dfm['frames_win_ratio_diff_1_year']=dfm['p1_frames_win_ratio_1_year']-dfm['p2_frames_win_ratio_1_year']
dfm['frames_win_ratio_diff_3_years']=dfm['p1_frames_win_ratio_3_years']-dfm['p2_frames_win_ratio_3_years']

In [17]:
selected_features = ['matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']

In [18]:
X = dfm[selected_features]
y = dfm['win_percentage']

In [19]:
tscv = TimeSeriesSplit(n_splits=5)

rmse_list = []
train_sizes = []

for train_index, test_index in tscv.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Test size:  {len(test_index)}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    rmse_list.append(rmse)
    train_sizes.append(len(train_index))

    print(f"Fold RMSE: {rmse} \n")

  Train size: 705
  Test size:  702
Fold RMSE: 0.31613796704969827 

  Train size: 1407
  Test size:  702
Fold RMSE: 0.28431846477954076 

  Train size: 2109
  Test size:  702
Fold RMSE: 0.26690287503319443 

  Train size: 2811
  Test size:  702
Fold RMSE: 0.25236598399245175 

  Train size: 3513
  Test size:  702
Fold RMSE: 0.3055048787107596 



In [20]:
weighted_rmse = np.average(rmse_list, weights=train_sizes)
print(f"Weighted Avg RMSE:   {weighted_rmse:.4f}")

Weighted Avg RMSE:   0.2815


The WRMSEs are very similar, suggesting that experience is not playing much of a factor.

### Linear regression with player elo ratings and features based on player statistic differences

In [21]:
selected_features = ['player1_elo','player2_elo','matches_win_ratio_diff','frames_win_ratio_diff','frames_win_ratio_diff_1_year','frames_win_ratio_diff_3_years']

In [22]:
X = dfm[selected_features]
y = dfm['win_percentage']

In [23]:
tscv = TimeSeriesSplit(n_splits=5)

rmse_list = []
train_sizes = []

for train_index, test_index in tscv.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Test size:  {len(test_index)}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    rmse_list.append(rmse)
    train_sizes.append(len(train_index))

    print(f"Fold RMSE: {rmse} \n")

  Train size: 705
  Test size:  702
Fold RMSE: 0.3105672643918638 

  Train size: 1407
  Test size:  702
Fold RMSE: 0.2585491259439854 

  Train size: 2109
  Test size:  702
Fold RMSE: 0.2558405616274821 

  Train size: 2811
  Test size:  702
Fold RMSE: 0.23931395844201916 

  Train size: 3513
  Test size:  702
Fold RMSE: 0.30257946927964713 



In [24]:
weighted_rmse = np.average(rmse_list, weights=train_sizes)
print(f"Weighted Avg RMSE:   {weighted_rmse:.4f}")

Weighted Avg RMSE:   0.2710


As expected, adding the elo rating points is making the model perform better and similar to fully elo based calculations.

### Linear regression on all available features

In [25]:
features_to_remove = ['player1', 'player2', 
                      'score1','score2','match_result','win_percentage','tournament_id']
X = df.drop(columns=features_to_remove) 
y = df['win_percentage']

In [26]:
tscv = TimeSeriesSplit(n_splits=5)

rmse_list = []
train_sizes = []

for train_index, test_index in tscv.split(X):
    print(f"  Train size: {len(train_index)}")
    print(f"  Test size:  {len(test_index)}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    rmse_list.append(rmse)
    train_sizes.append(len(train_index))

    print(f"Fold RMSE: {rmse} \n")

  Train size: 705
  Test size:  702
Fold RMSE: 0.3162265676042055 

  Train size: 1407
  Test size:  702
Fold RMSE: 0.26094298212127404 

  Train size: 2109
  Test size:  702
Fold RMSE: 0.2607898793427507 

  Train size: 2811
  Test size:  702
Fold RMSE: 0.2405724457793567 

  Train size: 3513
  Test size:  702
Fold RMSE: 0.3035881133538027 



In [27]:
weighted_rmse = np.average(rmse_list, weights=train_sizes)
print(f"Weighted Avg RMSE:   {weighted_rmse:.4f}")

Weighted Avg RMSE:   0.2734
